# baseline v9 — SSAFY 16기 2회차 AI 챌린지 (9/21~9/28)

**위에서 아래로 한 번씩 실행하면 제출 파일까지 나온다.**

## 실제 데이터로 확인한 것

| | |
|---|---|
| train / test | **6714 / 6714**건 (1차 test 5074보다 32% 큼) |
| dev | 2683건, 라벨 아님(5명 응답) |
| 정답 분포 | a 1700 · b 1644 · c 1716 · d 1654 — 균등 |
| 원본 해상도 | 짧은 변 **720px** (720×960 = 691k 픽셀) |
| 질문 유형 | 상호·이름 25.8% / 문구·판독 21.4% / 가격·할인 13.7% / 기타 10.9% … |
| **보기 유사도** | **1글자 차이 25.9%, 2글자 이하 44.8%** |

1차(재활용품)는 개수 세기·재질 판별이 중심이었다. 2차는 **전부 이미지 속 글자를 읽는 문제**다.
카운팅은 0.2%, 색상은 0.4%뿐이다.

## 그래서 해상도가 1순위다

```
공식 베이스라인 384² = 147k 픽셀 = 원본의 21%   <- 간판 글자가 뭉개진다
512²           = 262k = 38%
768²           = 590k = 85%    (권장 시작점)
1024²          = 1049k = 152%  (업샘플링, 토큰 2배)
```

보기의 절반이 2글자 이내로 갈리는데 원본의 21%로 줄이면 애초에 구분이 불가능하다.
**다른 어떤 튜닝보다 먼저 512 / 768 / 1024 를 비교하라.**

`min_pixels` 와 `max_pixels` 를 같은 값으로 묶으면 작은 이미지를 억지로 키워 토큰만 낭비하므로,
하한은 낮게 두고 상한만 건다.

## 1차(v4, 0.91407 / 67등) 대비 바뀐 것

| 항목 | v4 (1차) | v8 |
|---|---|---|
| 모델 | Qwen2.5-VL-**32B** | **3B** (공식 베이스라인) / 7B 전환 가능 |
| 해상도 | min=max 로 고정 | **하한/상한 분리, 768 기본, 최우선 실험 변수** |
| 손실 | 정답 위치에서 **전체 vocab** CE | **choice_ce** — a/b/c/d 4개 로짓만 |
| 학습 보기 순서 | 고정 | **셔플** (위치 편향 제거) |
| qtype | 카운팅·재질 | **실측 기반 11종** (기타 10.9%) |
| 환경 | 코랩 전용 | `OFFLINE` 플래그로 코랩/데스크탑 겸용 |
| 저장 경로 | 공용 | `WHO`로 팀원별 분리 |
| dtype | T4라 fp16 강제 | GPU 보고 bf16 자동 |

유지한 것: 로짓 스코어링 추론, 4-way TTA, 홀드아웃 검증, 제로샷 기준점, 중간 저장 재개.

---

### 순서

`0 설정` → `1 설치` → `2 임포트` → `3 데이터·dev` → `4 EDA` → `5 모델` → `6 프롬프트`
→ `7 추론함수` → `8 제로샷` → `9 Dataset` → `10 학습` → `11 평가·TTA` → `12 제출`

학습을 건너뛰고 추론만 할 거면 `0~7` 실행 후 **부록 A(어댑터 로드)** 로.


## 0. 설정

**여기만 바꿔가며 실험한다.** 한 번에 하나씩만 바꿀 것.

In [ ]:
# ============================================================
# 0. 설정
# ============================================================

WHO        = ""
assert WHO, "0번 셀 WHO 에 본인 이름을 넣어주세요"


OFFLINE    = False        # False: 코랩(모델 인터넷) / True: 데스크탑(로컬 폴더)
MODEL_SIZE = "7b"         # "3b" or "7b" — 09/21 제로샷 비교로 7B 확정 (0.912 vs 0.860)

# 손실 방식 — 이번 대회의 핵심 비교 대상
#   "choice_ce" : a/b/c/d 4개 로짓에만 CE  (추론과 목적함수 일치)
#   "mask"      : 정답 위치 전체 vocab CE  (v4 방식, 기준선 비교용)
LOSS_MODE = "choice_ce"

SHUFFLE_CHOICES = True    # 학습 시 보기 순서 셔플 (위치 편향 제거)

# ── 해상도: 이번 대회에서 가장 큰 레버 ────────────────────────────
# 원본 사진은 짧은 변 720px (720x960 = 691k 픽셀).
# 공식 베이스라인 384²=147k 는 원본의 21%로, 간판 글자가 뭉개진다.
# train 보기의 25.9%가 1글자 차이, 44.8%가 2글자 이하 차이라 판독이 곧 점수다.
#   384 -> 147k (21%)   512 -> 262k (38%)
#   768 -> 590k (85%)  1024 -> 1049k (152%, 업샘플링)
# min 과 max 를 같게 묶으면 작은 이미지를 억지로 키워 토큰만 낭비하므로 분리한다.
IMAGE_SIZE = 768          # 픽셀 예산의 제곱근. 768 / 1024 를 반드시 비교할 것
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = IMAGE_SIZE * IMAGE_SIZE

TRAIN_N    = 6000         # train 전체 6714건. 빠른 검증은 500
VALID_N    = 500          # 홀드아웃 검증셋

# dev.csv: 실측 결과 4명 이상 일치가 **0건**이다(최대 3명, 2026건/75.5%).
# 즉 dev는 '기준 미달로 폐기된 풀'이라 라벨 신뢰도가 train보다 명백히 낮다.
# 기본은 사용 안 함. 3명 일치를 추가 학습에 넣는 실험을 할 때만 켠다.
USE_DEV       = None      # None / "valid"(보조 검증) / "train"(추가 학습)
DEV_MIN_AGREE = 3
EPOCHS     = 1
BATCH      = 1
GRAD_ACCUM = 8
LR         = 1e-4         # choice_ce는 신호가 선명해 v4(5e-5)보다 올려도 된다
LORA_R     = 16           # 3B는 여유 있음. 32까지 실험 가능
EVAL_BATCH = 4            # 7B/768 기준 4. 해상도 올리면 2 또는 1까지 줄인다
SEED       = 42

N_TTA = 3                 # 제출 시 TTA 수 (0=단독, 1=2-way, 3=4-way)

# --- 경로 ---
DRIVE_DIR = "/content/drive/MyDrive"           # 코랩 드라이브 마운트 지점
ZIP_HINT  = "ssafy-16-2"                    # 드라이브에서 찾을 zip 파일명 일부
ZIP_PW    = ""                              # 배포 zip 암호. 없으면 빈 문자열
DATA_DIR  = "/content/data"
OUT_DIR   = f"{DRIVE_DIR}/ssafy_ai/{WHO}"

if OFFLINE:                                  # 대회 당일 데스크탑
    DATA_DIR = "./data"
    OUT_DIR  = f"./runs/{WHO}"

_HF    = {"3b": "Qwen/Qwen2.5-VL-3B-Instruct",
          "7b": "Qwen/Qwen2.5-VL-7B-Instruct"}
_LOCAL = {"3b": "downloads/models/Qwen2.5-VL-3B-Instruct",
          "7b": "downloads/models/Qwen2.5-VL-7B-Instruct"}
MODEL_PATH = _LOCAL[MODEL_SIZE] if OFFLINE else _HF[MODEL_SIZE]

# 실험 태그 — 체크포인트/제출 파일명에 붙어 나중에 구분된다
TAG = (f"{MODEL_SIZE}_{LOSS_MODE}_{'shuf' if SHUFFLE_CHOICES else 'noshuf'}"
       f"_img{IMAGE_SIZE}_r{LORA_R}_e{EPOCHS}")

print("MODEL :", MODEL_PATH)
print("TAG   :", TAG)
print("OUT   :", OUT_DIR)

## 1. 패키지 (코랩만)

데스크탑은 `download_libs_models.ipynb` 로 이미 설치돼 있으니 자동으로 건너뛴다.
코랩에서 설치가 돌았으면 **런타임 - 세션 다시 시작** 후 0번부터 다시.

In [ ]:
import sys, subprocess

if OFFLINE:
    print("오프라인 환경 — 설치 건너뜀")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.49.0", "accelerate>=0.34.2", "peft>=0.13.2",
                    "bitsandbytes>=0.43.3", "pillow", "pandas", "--upgrade"], check=True)
    print("설치 완료 — 런타임 - 세션 다시 시작 후 0번부터 다시 실행")

## 2. 임포트 + 환경 확인

`bf16: True` 면 L4/A100/5060Ti, `False` 면 T4다. 자동으로 fp16 + GradScaler 로 전환된다.

In [ ]:
import os, math, random, zipfile, glob
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"

# GPU에 맞는 dtype 자동 선택
USE_BF16   = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
AMP_DTYPE  = torch.bfloat16 if USE_BF16 else torch.float16
USE_SCALER = not USE_BF16

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU      :", torch.cuda.get_device_name() if torch.cuda.is_available() else "없음")
print("VRAM(GB) :", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1)
      if torch.cuda.is_available() else "-")
print("AMP      :", AMP_DTYPE, "| GradScaler:", USE_SCALER)
assert torch.cuda.is_available(), "GPU 런타임인지 확인 (런타임 - 런타임 유형 변경)"

## 3. 데이터

드라이브에 올린 zip을 `/content/data` 로 푼다.
**이미지를 드라이브에서 직접 읽으면 안 된다** — 네트워크 마운트라 학습/추론이 몇 배 느려진다.
세션이 끊기면 이 셀만 다시 실행.

In [ ]:
# ============================================================
# 3. 데이터
# ============================================================
import os, glob, time, subprocess, shutil

if not OFFLINE:
    from google.colab import drive
    MNT = "/content/drive"
    if not os.path.ismount(MNT):
        # 마운트 전에 로컬로 생겨버린 가짜 폴더 정리 (진짜 마운트면 여기 안 들어옴)
        if os.path.exists(MNT) and os.listdir(MNT):
            shutil.rmtree(MNT)
            print("가짜 로컬 폴더 정리")
        drive.mount(MNT)

    # 마운트 검증 먼저. 이 전에는 절대 makedirs 하지 않는다
    # 휴지통(.Trash)에 옛 zip이 있어도 안 잡히게 MyDrive 를 우선 본다
    roots = ([f"{MNT}/MyDrive"] if os.path.isdir(f"{MNT}/MyDrive")
             else [f"{MNT}/{d}" for d in os.listdir(MNT)])
    print("찾는 위치 :", roots)
    hits = [p for r in roots for p in glob.glob(f"{r}/**/*.zip", recursive=True)
            if ZIP_HINT in os.path.basename(p)]
    assert hits, "zip 못 찾음. 드라이브 루트: " + str({r: os.listdir(r)[:8] for r in roots})

    ZIP_PATH  = hits[0]
    DRIVE_DIR = os.path.dirname(ZIP_PATH)            # zip 이 있는 곳 = 내 드라이브
    OUT_DIR   = f"{DRIVE_DIR}/ssafy_ai/{WHO}"
    print("DRIVE_DIR :", DRIVE_DIR)
    print("zip       :", ZIP_PATH, f"({os.path.getsize(ZIP_PATH)/1e9:.2f}GB)")

    # 검증 끝났으니 이제 폴더 생성
    os.makedirs(OUT_DIR,  exist_ok=True)
    os.makedirs(DATA_DIR, exist_ok=True)

    if not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
        assert ZIP_PW, "0번 셀 ZIP_PW 에 암호를 넣어주세요 (단톡 공지 참고)"
        t0 = time.time()
        subprocess.run("apt-get -qq install -y p7zip-full", shell=True, check=False)
        cmd = ["7z", "x", "-y", f"-o{DATA_DIR}", f"-p{ZIP_PW}", ZIP_PATH]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise SystemExit("압축 해제 실패 — ZIP_PW 확인\n" + (r.stdout or "")[-400:])
        print(f"해제 완료 {(time.time()-t0)/60:.1f}분")

if not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
    inner = [d for d in glob.glob(os.path.join(DATA_DIR, "*"))
             if os.path.isdir(d) and os.path.exists(os.path.join(d, "train.csv"))]
    assert inner, f"train.csv 못 찾음: {os.listdir(DATA_DIR)}"
    DATA_DIR = inner[0]
    print("DATA_DIR 재설정:", DATA_DIR)

print("\ncsv :", sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".csv")))
for d in ["train", "test", "dev"]:
    p = os.path.join(DATA_DIR, d)
    print(f"{d:6s}:", len(os.listdir(p)) if os.path.isdir(p) else "없음")

In [ ]:
train_all = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df   = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
print("컬럼 :", list(train_all.columns))

# path 컬럼을 DATA_DIR 기준 절대경로로
def fix_path(p):
    p = str(p)
    return p if os.path.isabs(p) else os.path.join(DATA_DIR, p)

train_all["path"] = train_all["path"].map(fix_path)
test_df["path"]   = test_df["path"].map(fix_path)
assert os.path.exists(train_all["path"].iloc[0]), f"이미지 경로 확인: {train_all['path'].iloc[0]}"

# 셔플 후 홀드아웃 분리
train_all = train_all.sample(frac=1, random_state=SEED).reset_index(drop=True)
valid_df  = train_all.iloc[:VALID_N].reset_index(drop=True)
fit_df    = train_all.iloc[VALID_N:VALID_N + TRAIN_N].reset_index(drop=True)

print(f"\n학습 {len(fit_df)} / 홀드아웃 {len(valid_df)} / test {len(test_df)}")
print("정답 분포:", train_all['answer'].value_counts().sort_index().to_dict())

### dev.csv — 기대와 달리 '폐기된 풀'이다

`dev.csv` 의 `answer1~5` 는 정답이 아니라 **교육생 5명이 각자 푼 응답**이다.
대회 설명은 "5명 중 4명 이상 일치한 퀴즈만 정답으로 채택, 기준 미달은 폐기"라고 했으니
dev에서 4명 이상 일치한 행은 train과 같은 품질의 라벨이 되리라 기대할 수 있다.

**실측하면 그런 행이 0건이다.**

```
0명 일치     2건  ( 0.1%)
1명 일치     6건  ( 0.2%)
2명 일치   649건  (24.2%)
3명 일치  2026건  (75.5%)   <- 최대
4명 이상    0건  ( 0.0%)
```

즉 dev는 정확히 **기준 미달로 폐기된 문항 모음**이다. 3명 일치를 라벨로 쓰면
train보다 신뢰도가 확실히 낮은 데이터를 섞는 셈이라 기본값은 사용 안 함(`USE_DEV=None`)이다.

그래도 쓸모는 있다 — **"사람도 헷갈린 어려운 문항" 2683건**이므로, 여기서 모델이 무너지는
유형을 보면 오답 분석의 방향이 잡힌다. 다만 절대 정확도 수치는 믿지 말 것.
`USE_DEV="train"` 으로 3명 일치를 학습에 섞는 실험은 해볼 가치가 있고, 홀드아웃으로 판정한다.

In [ ]:
from collections import Counter

_L       = ["a", "b", "c", "d"]
dev_path = os.path.join(DATA_DIR, "dev.csv")
dev_use  = None

if USE_DEV and os.path.exists(dev_path):
    dev = pd.read_csv(dev_path)
    dev["path"] = dev["path"].map(fix_path)
    acols = [c for c in dev.columns if c.startswith("answer")]
    print("dev 응답 컬럼:", acols)

    def vote(row):
        vals = [str(row[c]).strip().lower() for c in acols
                if pd.notna(row[c]) and str(row[c]).strip().lower() in _L]
        if not vals:
            return "", 0
        lab, n = Counter(vals).most_common(1)[0]
        return lab, n

    v   = dev.apply(lambda r: pd.Series(vote(r), index=["answer", "n_agree"]), axis=1)
    dev = pd.concat([dev.drop(columns=acols), v], axis=1)

    print("dev 전체      :", len(dev))
    print("일치 인원 분포:", dev["n_agree"].value_counts().sort_index().to_dict())

    dev_use = dev[dev["n_agree"] >= DEV_MIN_AGREE].reset_index(drop=True)
    print(f"{DEV_MIN_AGREE}명 이상 일치 : {len(dev_use)}건  "
          f"({len(dev_use)/max(1,len(dev)):.1%})")

    if USE_DEV == "train" and len(dev_use):
        fit_df = pd.concat([fit_df, dev_use[fit_df.columns]], ignore_index=True)
        fit_df = fit_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
        print("-> 학습 데이터에 합침. 학습:", len(fit_df))
    elif USE_DEV == "valid":
        print("-> dev_use 를 보조 검증셋으로 사용 가능: evaluate(dev_use, 'dev')")
else:
    print("dev 사용 안 함")

## 4. EDA — 질문 유형 분포

**대회 첫날 여기서 시간을 써라.** 어떤 유형이 많은지 알아야 프롬프트와 오답 분석의 방향이 잡힌다.

아래 규칙은 2차 dev 샘플(상호명·가격·전화번호·강의실 번호·간판 문구)을 보고 짠 초안이다.
**출력되는 '기타' 샘플을 읽고 반드시 직접 보강할 것.** '기타'가 20%를 넘으면 규칙이 데이터를 못 따라가는 것.

In [ ]:
def qtype(q):
    """train 6714건으로 튜닝한 규칙. 순서 중요 — 위에서부터 먼저 걸린다."""
    q = str(q)
    if any(k in q for k in ["포함되지 않", "아닌 것", "없는 것", "없는 서비스", "제공하지 않",
                            "해당하지 않", "옳지 않", "올바르지 않", "잘못된", "틀린", "불가능한"]):
        return "부정형"                       # 보기를 전부 읽어야 해서 가장 잘 틀린다
    if any(k in q for k in ["가격", "얼마", "요금", "금액", "원인가", "할인"]):
        return "가격·할인"
    if any(k in q for k in ["전화번호", "연락처", "번호는", "몇 층", "호실", "번지", "몇 번"]):
        return "번호·연락처"
    if any(k in q for k in ["시간", "언제", "기간", "날짜", "요일", "영업", "시부터", "발행일", "마감"]):
        return "시간·일정"
    if any(k in q for k in ["주소", "위치", "어디", "왼쪽", "오른쪽", "방면", "출구", "방향"]):
        return "위치·주소"
    if any(k in q for k in ["상호명", "브랜드", "가게 이름", "업체", "회사", "이름은", "명은", "제목"]):
        return "상호·이름"
    if any(k in q for k in ["메뉴", "음식", "제품", "모델", "상품"]):
        return "메뉴·상품"
    if any(k in q for k in ["적힌", "적혀", "쓰여", "쓰인", "문구", "글자",
                            "표기", "표시", "내용", "설명", "의미"]):
        return "문구·판독"
    if any(k in q for k in ["몇 개", "개수", "갯수"]):
        return "카운팅"
    if "색" in q:
        return "색상"
    return "기타"

train_all["qtype"] = train_all["question"].apply(qtype)
vc = train_all["qtype"].value_counts()
print(vc.to_string())
print(f"\n'기타' 비중: {vc.get('기타', 0)/len(train_all):.1%}")
print("참고 — train 6714건 실측: 상호·이름 25.8 / 문구·판독 21.4 / 가격·할인 13.7")
print("       기타 10.9 / 위치·주소 8.2 / 번호·연락처 5.7 / 메뉴·상품 4.9")
print("       시간·일정 4.8 / 부정형 4.1 / 색상 0.4 / 카운팅 0.2 (%)")
print("=> 1차(재활용품)의 카운팅·재질은 사실상 사라졌고, 전부 '글자 읽기' 문제다.\n")

print("--- 기타 샘플 20개 (읽고 규칙 보강 여지 확인) ---")
for q in train_all.loc[train_all["qtype"] == "기타", "question"].head(20):
    print(" -", q)

### 보기가 얼마나 비슷한가 — 해상도 결정 근거

2차는 `김진형치과 / 김태형치과 / 김준형치과 / 김준희치과` 처럼 **한두 글자만 다른 보기**가 나온다.
이런 문항 비중이 높으면 `IMAGE_SIZE` 를 올리는 것이 다른 어떤 튜닝보다 효과가 크다.

In [ ]:
import itertools

def _lev(x, y):
    if x == y: return 0
    m, n = len(x), len(y)
    prev = list(range(n + 1))
    for i in range(1, m + 1):
        cur = [i] + [0] * n
        for j in range(1, n + 1):
            cur[j] = min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (x[i-1] != y[j-1]))
        prev = cur
    return prev[n]

def min_pair_diff(row):
    """보기 4개 중 가장 비슷한 두 개의 편집거리 (작을수록 정밀 판독 필요)"""
    o = [str(row[l]) for l in ["a", "b", "c", "d"]]
    return min(_lev(x, y) for x, y in itertools.combinations(o, 2))

_s = train_all.sample(min(1000, len(train_all)), random_state=SEED).copy()
_s["dmin"] = _s.apply(min_pair_diff, axis=1)
print("보기 최소 편집거리 분포:", _s["dmin"].value_counts().sort_index().head(6).to_dict())
print(f"1글자 이하: {(_s.dmin<=1).mean():.1%} | 2글자 이하: {(_s.dmin<=2).mean():.1%}"
      f" | 3글자 이하: {(_s.dmin<=3).mean():.1%}")
print("참고 — train 실측: 1글자 25.9% / 2글자 이하 44.8% / 3글자 이하 56.4%")
print("=> 보기 절반이 2글자 이내로 갈린다. 해상도가 다른 어떤 튜닝보다 먼저다.\n")

print("--- 1글자 차이 문항 3개 ---")
for _, r in _s.nsmallest(3, "dmin").iterrows():
    print(f"Q: {r['question']}")
    print(f"   a){r['a']} | b){r['b']} | c){r['c']} | d){r['d']}  -> {r['answer']}")

In [ ]:
# 이미지 한 장 눈으로 확인 (해상도/화질 감 잡기)
_r = train_all.iloc[0]
_img = Image.open(_r["path"]).convert("RGB")
print("이미지 크기:", _img.size)
print("질문:", _r["question"])
print(f"(a) {_r['a']}  (b) {_r['b']}  (c) {_r['c']}  (d) {_r['d']}  -> 정답 {_r['answer']}")
_img.resize((_img.width // 3, _img.height // 3))

## 5. 모델 / Processor

3B는 약 7.5GB. 코랩에서 처음 받으면 5~10분.

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, get_linear_schedule_with_warmup
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoVLM
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

if OFFLINE:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=AMP_DTYPE,
)

processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    min_pixels=MIN_PIXELS,      # 하한만 낮게 — 작은 이미지를 억지로 키우지 않는다
    max_pixels=MAX_PIXELS,      # 상한만 걸어 원본 비율·해상도를 최대한 보존
    trust_remote_code=True,
)
tok = processor.tokenizer

# 실제로 몇 픽셀/몇 토큰이 들어가는지 확인 (OOM·속도 예측용)
_im = Image.open(train_all["path"].iloc[0]).convert("RGB")
_pv = processor.image_processor(images=_im, return_tensors="pt")
_ntok = _pv["pixel_values"].shape[0] // 4      # 2x2 merge
print(f"원본 {_im.size} -> 이미지 토큰 약 {_ntok}개  (MAX_PIXELS={MAX_PIXELS:,})")

base_model = AutoVLM.from_pretrained(
    MODEL_PATH, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_R * 2, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 6. 프롬프트 + 정답 토큰 id

`assert` 가 통과해야 아래 로짓 스코어링과 choice_ce가 성립한다.

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

def build_messages(question, opts, img, answer=None):
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": build_mc_prompt(question, *opts)},
        ]},
    ]
    if answer is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return msgs

LETTERS    = ["a", "b", "c", "d"]
LETTER_IDS = [tok.encode(l, add_special_tokens=False)[0] for l in LETTERS]
LETTER_T   = torch.tensor(LETTER_IDS, device=device)
IM_END_ID  = tok.convert_tokens_to_ids("<|im_end|>")

print("LETTER_IDS:", LETTER_IDS, "->", [tok.decode([i]) for i in LETTER_IDS])
assert [tok.decode([i]) for i in LETTER_IDS] == LETTERS, "토큰 분해 확인 필요"
assert len(set(LETTER_IDS)) == 4

## 7. 추론 함수 = 로짓 스코어링

`generate()` 대신 프롬프트 마지막 위치의 logits에서 a/b/c/d 4개만 꺼내 softmax.
파싱 실패가 구조적으로 불가능하고, forward 1번이라 빠르며, 확률이 남아 TTA/앙상블에 재활용된다.

`rotate=k` 는 보기를 왼쪽으로 k칸 회전해 추론하고, 확률은 원래 보기 기준으로 되돌린다.

In [ ]:
@torch.no_grad()
def predict_probs(df, batch_size=None, desc="predict", rotate=0,
                  save_path=None, save_every=500):
    batch_size = batch_size or EVAL_BATCH
    model.eval()
    tok.padding_side = "left"

    start, all_probs = 0, []
    if save_path and os.path.exists(save_path):          # 중단 지점부터 재개
        done = torch.load(save_path)
        all_probs, start = [done], done.shape[0]
        print(f"이어서 진행: {start}건 완료됨")

    for s in tqdm(range(start, len(df), batch_size), desc=desc):
        chunk = df.iloc[s:s + batch_size]
        texts, images = [], []
        for _, row in chunk.iterrows():
            img  = Image.open(row["path"]).convert("RGB")
            opts = [row["a"], row["b"], row["c"], row["d"]]
            if rotate:
                opts = opts[rotate:] + opts[:rotate]
            msgs = build_messages(row["question"], opts, img, answer=None)
            texts.append(processor.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True))
            images.append(img)

        enc = processor(text=texts, images=images, padding=True,
                        return_tensors="pt").to(model.device)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = model(**enc).logits[:, -1, :]
        probs = logits[:, LETTER_IDS].float().softmax(-1).cpu()

        if rotate:                                        # 원래 보기 순서로 복원
            back = torch.zeros_like(probs)
            for j in range(4):
                back[:, (j + rotate) % 4] = probs[:, j]
            probs = back
        all_probs.append(probs)

        if save_path and (s // batch_size) % max(1, save_every // batch_size) == 0:
            torch.save(torch.cat(all_probs), save_path)

    out = torch.cat(all_probs)
    if save_path:
        torch.save(out, save_path)
    return out


def probs_to_letters(probs):
    return [LETTERS[i] for i in probs.argmax(-1).tolist()]


def acc_of(probs, df):
    gold = df["answer"].astype(str).str.strip().str.lower().tolist()
    return float(np.mean([p == g for p, g in zip(probs_to_letters(probs), gold)]))


def evaluate(df, name="valid", tta=0):
    """tta=0 단독, 1 2-way, 3 4-way"""
    probs = predict_probs(df, desc=f"eval[{name}]")
    for r in range(1, tta + 1):
        probs = probs + predict_probs(df, desc=f"eval[{name}]+r{r}", rotate=r)
    a = acc_of(probs, df)
    print(f"[{name}] accuracy = {a:.4f}  (n={len(df)}, tta={tta})")
    return a, probs


def report_by_qtype(df, probs, name=""):
    """유형별 정확도 — 어느 유형이 약한지가 다음 실험의 방향이 된다"""
    d = df.copy().reset_index(drop=True)
    d["correct"] = pd.Series(probs_to_letters(probs)) == \
                   d["answer"].astype(str).str.strip().str.lower()
    d["qtype"] = d["question"].apply(qtype)
    print(f"--- {name} 유형별 ---")
    print(d.groupby("qtype")["correct"].agg(["mean", "count"]).round(3)
           .sort_values("mean").to_string())
    return d

## 8. 제로샷 기준점 — **파인튜닝 전에 반드시**

이 숫자가 없으면 LoRA가 성능을 올렸는지 내렸는지 알 수 없다.
학습 후 이 값을 못 넘으면 학습 설정이 잘못된 것이다.

In [ ]:
zs_acc, zs_probs = evaluate(valid_df, f"{MODEL_SIZE}-zeroshot")
_ = report_by_qtype(valid_df, zs_probs, "zeroshot")

In [ ]:
# 제로샷 test 제출 (학습 없이). 필요할 때만 True 로 바꿔서 실행 — 7B 기준 약 36분
RUN_ZS_SUBMIT = False

if RUN_ZS_SUBMIT:
    zs_test = predict_probs(test_df, desc="test-zeroshot",
                            save_path=f"{OUT_DIR}/test_zeroshot_{MODEL_SIZE}_img{IMAGE_SIZE}.pt")

    sub_name   = f"sub_zeroshot_{MODEL_SIZE}_img{IMAGE_SIZE}_hold{zs_acc:.4f}.csv"
    submission = pd.DataFrame({"id": test_df["id"], "answer": probs_to_letters(zs_test)})
    submission.to_csv(f"/content/{sub_name}", index=False)
    submission.to_csv(f"{OUT_DIR}/{sub_name}", index=False)

    print(sub_name)
    print(submission["answer"].value_counts().sort_index().to_string())
else:
    print("제로샷 제출 건너뜀 (RUN_ZS_SUBMIT=False)")

## 9. Dataset / Collator — 보기 셔플 + choice_ce 라벨

**보기 셔플**: 학습할 때마다 (a)(b)(c)(d) 내용을 섞고 정답 글자도 따라 바꾼다.
모델이 "정답은 주로 c" 같은 위치 편향을 외우는 걸 막는다. epoch마다 순서가 달라져 증강 효과도 있다.
검증/테스트에는 적용하지 않는다(대신 추론 TTA로 처리).

collator가 내보내는 것
- `labels` : v4 방식(정답 위치만 남기고 -100) — `LOSS_MODE="mask"` 일 때 사용
- `ans_pos`, `choice_labels` : choice_ce 계산용

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, train=True, shuffle_choices=False):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.shuffle_choices = shuffle_choices
        self.epoch = 0                     # 학습 루프에서 매 epoch 갱신

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row  = self.df.iloc[i]
        img  = Image.open(row["path"]).convert("RGB")
        opts = [str(row[l]) for l in LETTERS]
        gold = str(row["answer"]).strip().lower() if self.train else None

        if self.train and self.shuffle_choices:
            rng   = random.Random((SEED * 1000003) ^ (self.epoch * 7919) ^ i)
            order = list(range(4))
            rng.shuffle(order)
            g_old = LETTERS.index(gold)
            opts  = [opts[k] for k in order]
            gold  = LETTERS[order.index(g_old)]     # 원래 정답이 옮겨간 새 위치

        msgs = build_messages(row["question"], opts, img, answer=gold)
        text = processor.apply_chat_template(msgs, tokenize=False,
                                             add_generation_prompt=False)
        return {"text": text, "image": img, "gold": gold}


@dataclass
class DataCollator:
    processor: Any

    def __call__(self, batch):
        self.processor.tokenizer.padding_side = "right"
        enc = self.processor(
            text=[b["text"] for b in batch],
            images=[b["image"] for b in batch],
            padding=True, return_tensors="pt",
        )
        labels  = torch.full_like(enc["input_ids"], -100)
        ans_pos = []
        for i in range(len(batch)):
            ids   = enc["input_ids"][i]
            valid = enc["attention_mask"][i].bool()
            ends  = (ids.eq(IM_END_ID) & valid).nonzero().flatten()
            p     = ends[-1].item() - 1          # 마지막 <|im_end|> 직전 = 정답 글자
            ans_pos.append(p)
            labels[i, p] = ids[p]

        enc["labels"]        = labels
        enc["ans_pos"]       = torch.tensor(ans_pos, dtype=torch.long)
        enc["choice_labels"] = torch.tensor(
            [LETTERS.index(b["gold"]) for b in batch], dtype=torch.long)
        return enc

### 검증 셀 — 건너뛰지 말 것

세 가지가 맞아야 한다.
1. loss 대상 토큰이 a/b/c/d 중 하나
2. 그 토큰이 `choice_labels` 와 일치
3. 셔플을 켰으면 원본과 다른 글자가 나오기도 해야 한다 — **0/8 이면 셔플이 안 걸린 것**

In [ ]:
_ds    = VQAMCDataset(fit_df.head(8), train=True, shuffle_choices=SHUFFLE_CHOICES)
_check = DataCollator(processor)([_ds[i] for i in range(8)])

n_changed = 0
for i in range(8):
    p    = _check["ans_pos"][i].item()
    tokn = tok.decode([_check["input_ids"][i][p]])
    cl   = LETTERS[_check["choice_labels"][i].item()]
    orig = str(fit_df.iloc[i]["answer"]).strip().lower()
    ok   = (tokn == cl) and (tokn in LETTERS)
    n_changed += (orig != cl)
    print(f"{i}: 원본={orig} 셔플후={cl} loss토큰='{tokn}' {'OK' if ok else '<<< 이상'}")

print(f"\n셔플로 정답 위치가 바뀐 샘플: {n_changed}/8  (SHUFFLE_CHOICES={SHUFFLE_CHOICES})")
assert all(tok.decode([_check['input_ids'][i][_check['ans_pos'][i].item()]]) in LETTERS
           for i in range(8))

## 10. 파인튜닝

**choice_ce**: 정답 위치 바로 앞 logits에서 a/b/c/d 4개만 뽑아 4-class cross entropy.
v4는 여기서 전체 vocab(약 15만)에 CE를 걸었는데, 정작 추론은 4개만 본다.
학습과 추론의 목적함수를 일치시키는 것이 이 변경의 전부다.

중간에 끊겨도 되도록 200스텝마다 어댑터를 저장한다.

In [ ]:
train_ds = VQAMCDataset(fit_df, train=True, shuffle_choices=SHUFFLE_CHOICES)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          collate_fn=DataCollator(processor), num_workers=2)

model.train()
model.config.use_cache = False
params    = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR)
num_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, max(1, int(num_steps * 0.03)), num_steps)
scaler    = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

CKPT_DIR    = f"{OUT_DIR}/lora_{TAG}"
global_step = 0

for epoch in range(EPOCHS):
    train_ds.epoch = epoch          # epoch마다 다른 셔플 순서
    running, seen = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch")

    for step, batch in enumerate(pbar, start=1):
        batch         = {k: (v.to(device) if torch.is_tensor(v) else v)
                         for k, v in batch.items()}
        ans_pos       = batch.pop("ans_pos")
        choice_labels = batch.pop("choice_labels")
        mask_labels   = batch.pop("labels")

        with torch.autocast("cuda", dtype=AMP_DTYPE):
            if LOSS_MODE == "mask":
                batch["labels"] = mask_labels
                raw_loss = model(**batch).loss
            else:
                out    = model(**batch)
                idx    = torch.arange(out.logits.size(0), device=device)
                # logits[t] 가 토큰[t+1] 을 예측하므로 ans_pos-1 위치를 본다
                chosen = out.logits[idx, ans_pos - 1, :].index_select(-1, LETTER_T)
                raw_loss = F.cross_entropy(chosen.float(), choice_labels)

        loss = raw_loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        running += raw_loss.item(); seen += 1

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step(); optimizer.zero_grad(set_to_none=True)
            global_step += 1
            pbar.set_postfix({"loss": f"{running/seen:.4f}"})
            running, seen = 0.0, 0
            if global_step % 200 == 0:
                model.save_pretrained(CKPT_DIR)

model.save_pretrained(CKPT_DIR)
processor.save_pretrained(CKPT_DIR)
print("Saved:", CKPT_DIR)

## 11. 평가 — 제로샷보다 올랐는가?

**안 올랐으면 제출하지 말고 원인부터.**
1. 9번 검증 셀 출력이 이상함
2. LR이 너무 큼 (1e-4 → 5e-5)
3. 학습 데이터가 너무 적음

In [ ]:
model.config.use_cache = True
ft_acc, ft_probs = evaluate(valid_df, TAG)
d = report_by_qtype(valid_df, ft_probs, TAG)

print(f"\n제로샷 {zs_acc:.4f} -> 파인튜닝 {ft_acc:.4f}  ({ft_acc - zs_acc:+.4f})")

### TTA 효과 측정

보기 순서를 회전시켜 여러 번 추론하고 확률을 더한다.
시간이 배수로 늘어나므로 **홀드아웃에서 실제로 오르는지 확인하고** 테스트에 적용한다.
여기 결과를 보고 0번 셀의 `N_TTA` 를 정한다.

In [ ]:
p0 = ft_probs
p1 = predict_probs(valid_df, desc="tta-r1", rotate=1)
p2 = predict_probs(valid_df, desc="tta-r2", rotate=2)
p3 = predict_probs(valid_df, desc="tta-r3", rotate=3)

hold = {
    "단독":      round(acc_of(p0, valid_df), 4),
    "2-way TTA": round(acc_of(p0 + p1, valid_df), 4),
    "4-way TTA": round(acc_of(p0 + p1 + p2 + p3, valid_df), 4),
}
for k, v in hold.items():
    print(f"{k:10s}: {v}")

torch.save({"p0": p0, "p1": p1, "p2": p2, "p3": p3}, f"{OUT_DIR}/hold_{TAG}.pt")
_ = report_by_qtype(valid_df, p0 + p1 + p2 + p3, f"{TAG} +4way")

## 12. 테스트 추론 + 제출 파일

중간 저장되므로 세션이 끊겨도 같은 셀을 다시 실행하면 이어서 간다.
파일명에 홀드아웃 점수가 박히므로 나중에 어느 제출이 뭐였는지 알 수 있다.

In [ ]:
probs = predict_probs(test_df, desc="test-r0", save_path=f"{OUT_DIR}/test_{TAG}_r0.pt")
for r in range(1, N_TTA + 1):
    probs = probs + predict_probs(test_df, desc=f"test-r{r}", rotate=r,
                                  save_path=f"{OUT_DIR}/test_{TAG}_r{r}.pt")

h = {0: hold["단독"], 1: hold["2-way TTA"], 3: hold["4-way TTA"]}.get(N_TTA, ft_acc)
sub_name = f"sub_{TAG}_tta{N_TTA+1}_hold{h}.csv"

submission = pd.DataFrame({"id": test_df["id"], "answer": probs_to_letters(probs)})
submission.to_csv(sub_name if OFFLINE else f"/content/{sub_name}", index=False)
submission.to_csv(f"{OUT_DIR}/{sub_name}", index=False)
print(sub_name)
print(submission["answer"].value_counts().sort_index().to_string())

### 제출 전 체크 — 통과해야 캐글에 올린다

In [ ]:
ss = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
print("행 수 일치 :", len(submission) == len(ss))
print("id 일치    :", (submission["id"].values == ss["id"].values).all())
print("결측       :", int(submission["answer"].isna().sum()))
ratio = submission["answer"].value_counts(normalize=True).sort_index()
print("분포       :", ratio.round(3).to_dict(), "| 최대 쏠림:", round(ratio.max(), 3))
# 실제 정답은 a~d 거의 균등하다. 최대 쏠림이 0.4를 넘으면 뭔가 잘못된 것.

---

## 부록 A. 세션이 끊겼다면 — 어댑터만 불러오기

`0~7` 셀까지 다시 실행한 뒤, **10번 학습 셀 대신** 아래를 실행한다.

In [ ]:
# model = PeftModel.from_pretrained(base_model, f"{OUT_DIR}/lora_{TAG}")
# model.eval()
# print("어댑터 로드:", f"{OUT_DIR}/lora_{TAG}")

## 부록 B. 실험 기록

캐글 제출 한도는 **팀 전체 공유**다. 홀드아웃에서 기존 최고를 넘을 때만 제출한다.
아래 표를 팀 공유 시트로 옮겨 쓸 것 — 발표 정성평가의 근거가 여기서 나온다.

| # | 담당 | 모델 | img | LOSS_MODE | 셔플 | dev | r | TTA | 홀드아웃 | 제출 | 리더보드 | 메모 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 01 | | 3b | 512 | choice_ce | O | - | 16 | 1 | | | | 기준선 |
| 02 | | 3b | **768** | choice_ce | O | - | 16 | 1 | | | | ★ 해상도 |
| 03 | | 3b | **1024** | choice_ce | O | - | 16 | 1 | | | | ★ 해상도 |
| 04 | | 3b | best | mask | O | - | 16 | 1 | | | | 손실 비교(v4 방식) |
| 05 | | 3b | best | choice_ce | X | - | 16 | 1 | | | | 셔플 효과 |
| 06 | | 3b | best | choice_ce | O | train | 16 | 1 | | | | dev 3명일치 추가 |
| 07 | | 3b | best | choice_ce | O | best | 32 | 1 | | | | LoRA rank |
| 08 | | 3b | best | choice_ce | O | best | best | 4 | | | | +4-way TTA |
| 09 | | 7b | best | choice_ce | O | best | best | 4 | | | | 모델 크기 |

**01~03(해상도)을 가장 먼저 돌려라.** 2차는 글자 판독이 주력이라 여기서 제일 크게 갈린다.
그다음 04·05로 choice_ce와 셔플의 기여분을 각각 측정한다 — 변수 하나씩만 바꿔야
"무엇이 얼마나 올렸다"고 발표에서 말할 수 있다.

### 대략의 소요 시간 (A100 / 3B / img 768 기준)

| 작업 | 건수 | 예상 |
|---|---|---|
| 제로샷 홀드아웃 | 500 | 3~5분 |
| 학습 1 epoch | 6000 | 40~70분 |
| test 추론 1회 | 6714 | 40~70분 |
| test 4-way TTA | 26856 | 3~5시간 |

해상도를 1024로 올리면 이미지 토큰이 약 1.8배 늘어 위 시간도 그만큼 늘어난다.
**4-way TTA는 마지막 하루에만 돌려라.** 중간 실험은 TTA 없이 홀드아웃으로만 비교한다.

### 첫날(9/21) 순서
1. `0~4` 실행 → EDA 확인 (실측값은 이미 주석에 넣어뒀다)
2. `TRAIN_N=500` 으로 한 바퀴 돌려 파이프라인 검증 — 에러는 여기서 다 잡는다
3. 해상도 512 / 768 비교 → 이긴 쪽으로 `TRAIN_N=6000` 본 학습
4. **저녁에 무조건 첫 제출** (점수 없이 2일차로 넘어가지 않는다)

### 2차에서 버릴 것 / 주울 것
- **버릴 것**: 카운팅 오버샘플링(2차엔 카운팅이 0.2%뿐), 레터박스 패딩(Qwen2.5-VL은
  dynamic resolution이라 검은 여백만 늘어난다), `min_pixels=max_pixels` 고정
- **주울 것**: 해상도 상향, '부정형'(4.1%) 전용 프롬프트, 오답 qtype 집중 분석,
  1글자 차이 문항만 따로 뽑아 오답률 확인
